# Robotics III (Lecture 04)

## 第一部分：运动规划 (Motion Planning)

运动规划的核心任务是在存在障碍物的环境中，寻找一条从起始状态到目标状态的无碰撞几何路径。

### 1. 问题定义 (Problem Formulation)
*   **配置空间 (Configuration Space, $\mathcal{C}\text{-space}$)**:
    *   定义：包含系统所有可能状态的集合，通常是 $\mathbb{R}^n$ 的子集。
    *   $\mathcal{C}_{free}$：所有无碰撞的合法状态集合。
    *   $\mathcal{C}_{obs}$：障碍物占据的状态集合。
    *   关系：$\mathcal{C} = \mathcal{C}_{free} \cup \mathcal{C}_{obs}$。
*   **规划目标**:
    *   给定 $\mathcal{C}_{free}$，起始状态 $q_{start}$ 和目标状态 $q_{goal}$。
    *   计算一系列连续动作（路径），使得机器人从 $start$ 移动到 $goal$ 且全程位于 $\mathcal{C}_{free}$ 中。

### 2. 碰撞检测建模 (Collision Modeling)
为了判断一个状态 $q$ 是否属于 $\mathcal{C}_{free}$，必须进行碰撞检测。
*   **模型简化**:
    *   **Visual Mesh**: 用于渲染，顶点多，计算昂贵。如Triangle Mesh（三角面片网格）。
    *   **Collision Mesh**: 用于物理计算，通常比 Visual Mesh 简单。如简化的球体组合。
*   **凸分解 (Convex Decomposition)**:
    *   **原因**: 凸多边形/凸多面体的碰撞检测远比非凸物体高效。
    *   **方法**:
        *   *Convex-Hull*: 单一凸包，效率最高但精度最低。
        *   *Exact Convex Decomposition*: NP-hard，生成过多碎片，不实用。
        *   *Approximate Convex Decomposition (ACD)*: 实用方案。在保证凹陷度（concavity）低于阈值的前提下，将网格分割为最少数量的凸块。

### 3. 基于采样的算法 (Sampling-based Algorithms)
不精确构建整个 $\mathcal{C}\text{-space}$，而是通过随机采样探索空间。
*   **优点**: 概率完备性 (Probabilistically complete)，适用于高维空间。
*   **缺点**: 无法保证最优性，在狭窄通道 (Narrow Passages) 表现不佳。

#### A. 概率路图法 (Probabilistic Roadmap, PRM)
适用于**多查询 (Multi-query)** 和静态场景。
1.  **构图阶段 (Map Construction)**:(非常耗时，所以适合场景不怎么变化，多次利用)
    *   在 $\mathcal{C}_{free}$ 中随机采样点。
    *   将采样点与其邻域内的点连接（需检测边是否碰撞）。
2.  **查询阶段 (Query)**:
    *   将 $q_{start}$ 和 $q_{goal}$ 接入图中。
    *   使用 Dijkstra 或 A* 算法搜索路径。
*   **采样优化策略**:
    *   *Uniform Sampling*: 均匀采样。
    *   *Gaussian Sampling*: 在障碍物边缘采样（高斯分布扰动），解决紧贴障碍物的路径问题。
    *   *Bridge Sampling*: 专门针对狭窄通道（Narrow Bridge），通过检测“障碍-空闲-障碍”模式来采样桥梁中点。

#### B. 快速扩展随机树 (Rapidly-exploring Random Trees, RRT)
适用于**单次查询 (Single-query)** 场景。
*   **核心逻辑**:
    1.  从 $q_{start}$ 开始生长一棵树 $\mathcal{T}$。
    2.  **采样**: 随机生成目标点 $q_{target}$（以概率 $\beta$ 选 $q_{goal}$，否则在 $\mathcal{C}_{free}$ 随机选，在exploration和exploition两者间权衡）。
    3.  **最近邻**: 在树中找到离 $q_{target}$ 最近的节点 $q_{near}$。
    4.  **扩展 (Extend)**: 从 $q_{near}$ 向 $q_{target}$ 移动步长 $\epsilon$，生成新节点 $q_{new}$。
        $$ q_{new} \leftarrow q_{near} + \frac{\epsilon}{|q_{target} - q_{near}|}(q_{target} - q_{near}) $$
    5.  **检测**: 若 $q_{new}$ 及路径无碰撞，将 $q_{new}$ 加入树。
*   **RRT-Connect**:
    *   双向生长：从 $start$ 和 $goal$ 同时生长两棵树。
    *   贪心策略：一棵树试图直接连接到另一棵树的新节点，大大加速收敛。

### 4. 路径后处理 (Shortcutting)
采样算法生成的路径通常是曲折、不自然的（jerky），且非最优。
*   **算法**:
    1.  在路径上随机取两点 $u, v$。
    2.  检测 $u, v$ 之间的直线是否无碰撞 ($Visible(u, v)$)。
    3.  若无碰撞，用直线代替原有的曲折路径。
    4.  重复迭代。

### 5. 工具库
*   **OMPL (Open Motion Planning Library)**: ROS MoveIt 的默认规划库，包含 RRT, RRT-Connect, PRM 等几何规划器。

From Path to Trajectory: 从q(s)到q(t)，加入对时间的考量。

## 第二部分：运动控制（Control）
### 一、 控制系统基础 (Fundamentals of Control Systems)

#### 1. 控制系统的定义与目标
*   **核心功能**：调节系统的行为，使其在面临外界干扰（disturbances）和模型不确定性（uncertainties）的情况下，依然能够跟随给定的参考信号（reference）。
*   **评价一个控制系统好坏的标准（有效性）**：
    *   **稳态性能**：到达参考状态时，具有很小的跟踪误差（Small tracking error）。
    *   **瞬态性能**：最小化瞬态响应时间（Minimize transient response），即响应速度要快。
    *   **稳定性**：保持稳定，尽量减少震荡（Remain stable with minimal oscillations）。

#### 2. 控制系统的主要组成部分
*   **传感器 (Sensor)**：用于获取系统当前状态（如：关节角度、速度、力矩/力）。
*   **控制器 (Controller)**：根据输入和误差计算控制指令。
*   **环境/系统 (Environment/System)**：包含执行器（actuator）和物理系统本身。

#### 3. 控制策略分类
*   **开环控制 (Open-loop / Feedforward 控制)**：
    *   控制信号仅基于参考信号计算，不考虑系统当前的实际输出。
*   **闭环控制 (Closed-loop / Feedback 控制)**：
    *   控制器利用**误差 (Error = 参考输入 - 实际输出)** 来动态调整控制信号。
    *   **优势**：对模型不确定性具有鲁棒性；能够抑制外部干扰和传感器噪声；能更精确地跟踪参考信号。

---

### 二、 单自由度系统的PD控制原理 (1-DOF System PD Control)

以简单的小车（Cart）为例，系统动力学方程为 $m\ddot{x} + d\dot{x} + kx = u$（其中 $d, k$ 为系统的阻尼和刚度，$u$ 为控制力）。为了方便分析，常将其简化为 $\ddot{x} = u$ 的纯惯性系统。

#### 1. P控制 (比例控制，Proportional Control)
*   **控制律**：$u = K_p x_e$，其中 $x_e = x_{ref} - x$ 是位置误差。
*   **作用与现象**：
    *   $K_p$ 较小：响应慢，稳态（跟踪）误差大。
    *   增加 $K_p$：响应变快，误差减小，但系统开始出现超调（overshoot）和震荡（oscillation）。
    *   $K_p$ 过大：会导致严重的系统震荡。

#### 2. D控制 (微分控制，Derivative Control) 及其引入
*   **引入目的**：解决纯P控制中“响应快与震荡大”的矛盾。
*   **作用机制**：
    *   增加“虚拟阻尼（virtual damping）”，有效减少超调和震荡。
    *   对误差的“变化率”做出反应，具有预测运动趋势的能力。
    *   使得系统能够同时实现快速且稳定的跟踪。
*   **PD控制律**：$u = K_p x_e + K_d \dot{x}_e$（即加上了速度反馈项）。
    *   增大 $K_d$ 可以增加阻尼。当 $K_d$ 适当时（中度阻尼或过阻尼），可以实现响应快、误差小且几乎无震荡的理想状态。

---

### 三、 PD参数整定理论与方法 (PD Control Tuning)

#### 1. 经验调参法与 Ziegler-Nichols 法
*   **经验法**：手动调整，观察系统响应。
*   **Ziegler-Nichols（启发式）整定法**：
    1.  先仅使用P控制，逐渐增大 $K_p$，直到系统出现**持续震荡**。
    2.  加入D项（初始值可设为 $K_d \approx 0.1 K_p$）。
    3.  在此基础上进行微调（Finetune）。

#### 2. 独立增加单个参数的影响 (调参规律表)
*   **增大 $K_p$**：上升时间减小（变快），超调增加，稳态误差减小，稳定性降低。
*   **增大 $K_i$**（积分项，此处作为补充对比）：上升时间减小，超调增加，**消除稳态误差**，稳定性降低。
*   **增大 $K_d$**：上升时间变化很小，**超调减小**，稳定时间减小，理论上不影响稳态误差，在 $K_d$ 较小阶段可提高稳定性。

#### 3. 基于标准二阶系统动力学的精确整定 (重点公式)
将物理系统转化为标准二阶系统模型：$\ddot{x} + 2\xi w_n \dot{x} + w_n^2 x = 0$
*   **系统参数决定系统行为**：
    *   $w_n$ (自然频率 Natural frequency)：影响系统的**响应时间**。
    *   $\xi$ (阻尼比 Damping ratio)：影响系统的**超调和震荡**。
*   **参数映射关系**：
    结合简化后的小车系统动力学 $\ddot{x} + K_d\dot{x} + K_px = 0$，通过对比系数可以得出：
    *   **$K_p = w_n^2$**
    *   **$K_d = 2\xi w_n$**
*   **物理直觉**：需要更快响应 $\rightarrow$ 增大 $w_n \rightarrow$ $K_p$ 变大；需要更大阻尼 $\rightarrow$ 增大 $\xi \rightarrow$ $K_d$ 变大。

---

### 四、 多自由度真实系统的PD控制 (Multi-DOF Systems)

在将上述理论应用到机器人（如机械臂、人形机器人）等具有多个关节（多自由度）的真实系统时，需要考虑复杂的动力学耦合与现实约束。

#### 1. 基本原则 (Rule of Thumb)
*   可以将单自由度的二阶系统整定方法**独立应用于机器人的每一个单关节**。
*   以人形机器人（如BeyondMimic）为例，独立建模每个关节：$I_j \ddot{q} = u$ （$I_j$ 常用反射惯量 reflected inertia 来估计）。
*   针对每个关节的PD参数设计：**$K_{p,j} = I_j w_n^2$** , **$K_{d,j} = 2 I_j \xi w_n$**。

#### 2. 人形机器人设计的特殊考量 (启发式参数设计)
*   **自然频率设定**：例如设为 $10Hz$ ($w_n = 20\pi$)。
    *   不能太硬（stiff）：防止接触地面/外界时损坏电机。
    *   不能太软（soft）：防止轨迹跟踪性能下降。
*   **阻尼比设定**：通常设为**过阻尼**（如 $\xi = 2$）。
    *   因为模型简化时只考虑了关节惯量而忽略了连杆惯量等，需要额外的阻尼来抑制实际产生的震荡。
*   **硬件与传感器限制**：
    *   **踝关节（Ankle joints）**：由于直接接触地面并承受冲击，需要更高的顺应性（compliance，即更软）。
    *   **传感器噪声**：虽然增大 $K_d$ 在理论上减少震荡，但实际中微分项会极大**放大速度传感器的噪声**。因此不能无限制地依赖高阻尼来消除震荡。
    *   **力矩限制**：所有参数调整必须考虑硬件的力矩饱和极限（Torque limits）。

#### 3. 闭环多自由度系统面临的核心问题
在多刚体动力学系统 $M(q)\ddot{q} + C(\dot{q}, q) = \tau$ 中，若对每个关节独立使用 $\tau = K_p q_e + K_d \dot{q}_e$，会面临两个问题：
1.  各关节动力学通过惯性矩阵 $M(q)$ 产生了**耦合（coupled）**。
2.  需要额外的控制力去抵消非线性项/偏移项 $C(\dot{q}, q)$，例如**重力补偿**。

---

### 五、 高阶控制策略与工程实践总结

#### 1. 基于模型的反馈线性化 (Feedback Linearization - 多用于机械臂)
相比人形机器人，机械臂（如Franka）的模型更精确，环境更可控，因此可以使用基于模型的系统化调参方法。
*   **控制律设计**：$\tau = M(q)u + C(\dot{q}, q)$
*   **推导过程**：代入动力学方程后，非线性和耦合项被抵消（解耦），系统被完美线性化为：$\ddot{q} = u$
*   **应用PD**：此时再应用 $u = K_p q_e + K_d \dot{q}_e$，即可在理想解耦状态下为每个关节独立调整PD参数。

#### 2. 工程调参技巧 (Gain Scheduling & Joint-specific tricks)
*   **增益调度（Gain scheduling）**：针对不同的任务（Task）使用不同的PD参数组合。
*   **肩部关节 (Shoulder joints) 策略**：通常需要调得**更硬（more stiff）**。
    *   原因：有效负载（payload）更大；转动惯量更大；基座关节极小的误差会被放大为末端执行器（end-effector）巨大的位置误差。
*   **腕部关节 (Wrist joints) 策略**：
    *   需要**更多的阻尼**以提供顺应性，并避免末端震荡。

#### 3. 真实世界系统调参标准流程 (Summary Workflow)
1.  **二阶近似理论计算（初始参数获取）**：
    *   将每个关节视为独立的二阶系统。
    *   定义期望的自然频率和阻尼。
    *   利用公式计算出初始的 $K_p, K_d$ 增益。
2.  **在实际硬件上进行微调（Finetune）**：
    *   补偿模型误差（如未建模的摩擦力等）。
    *   处理传感器噪声带来的影响（限制D增益）。
    *   考虑有效负载的变化和接触力的影响。
    *   严格遵守电机输出力矩极限和关节运动范围极限。